In [4]:
from pyspark.sql import SparkSession
from IPython.display import display


spark = SparkSession.builder.getOrCreate()

versions_df = spark.read.parquet("../data/processed/transform_versions.parquet")
display(versions_df.orderBy("generated_at_utc", ascending=False).toPandas())


,transform_version,generated_at_utc,target_width,target_height,resize_strategy,image_mode,image_format,partition_by,partitions,shuffle_partitions,rows_written,input_manifest_path,output_manifest_path,output_images_path
0,v1,2026-02-27T10:20:36.312200+00:00,224,224,keep_aspect_pad,L,PNG,"pathology,modality",16,16,4478,data/processed/manifest_bronze.parquet,data/processed/manifest_silver.parquet,data/processed/images_silver


In [5]:
preprocess_manifest_df = spark.read.parquet("../data/processed/manifest_bronze.parquet")
display(preprocess_manifest_df.select("pathology").groupBy("pathology").count().toPandas())

,pathology,count
0,Meningioma,874
1,_NORMAL,522
2,Astrocitoma,579
3,Neurocitoma,457
4,Schwannoma,465
5,Glioblastoma,204
6,Oligodendroglioma,224
7,Papiloma,237
8,Carcinoma,251
9,Tuberculoma,145


In [ ]:
split_versions_f = spark.read.parquet("../data/processed/split_versions.parquet")
display(split_versions_f.orderBy("generated_at_utc", ascending=False).toPandas().T)

,0
generated_at_utc,2026-02-23T08:06:38.805643+00:00
seed,42
train_ratio,0.7
val_ratio,0.15
test_ratio,0.15
rows_total,3957
rows_train,2752
rows_val,591
rows_test,614
input_manifest_path,data/processed/manifest_silver.parquet


In [3]:
training_manifest = spark.read.parquet("../data/processed/splits/current/training_manifest.parquet")
display(training_manifest.columns)
display(training_manifest.toPandas())

['image_id',
 'raw_path',
 'label_idx',
 'file_size',
 'is_valid',
 'processed_path',
 'processed_file_size',
 'orig_width',
 'orig_height',
 'new_width',
 'new_height',
 'channels',
 'transform_version',
 'pathology',
 'modality',
 'split_id',
 'split_seed',
 'split_ratios',
 'split']

,image_id,raw_path,label_idx,file_size,is_valid,processed_path,processed_file_size,orig_width,orig_height,new_width,new_height,channels,transform_version,pathology,modality,split_id,split_seed,split_ratios,split
0,e3d48efbb74908d45ed3ffdb1082b17d84da319d5b3927...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,8,57343,True,data/processed/images_silver/v1/pathology=Meni...,23163,630,630,224,224,1,v1,Meningioma,T1C+,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,train
1,5900c3ad48d7a86e0c1e9567673024035a4213bc3f5411...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,8,49838,True,data/processed/images_silver/v1/pathology=Meni...,27059,630,630,224,224,1,v1,Meningioma,T1C+,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,train
2,28cc83ade7df5c94c2c833360151566e84d0862342f46a...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,8,44996,True,data/processed/images_silver/v1/pathology=Meni...,19902,630,630,224,224,1,v1,Meningioma,T1C+,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,train
3,d2bf0cd96d86150dc12df40cfda35eb7c870956e3dbb47...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,8,56635,True,data/processed/images_silver/v1/pathology=Meni...,22897,630,630,224,224,1,v1,Meningioma,T1C+,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,train
4,25033cda90afb41dfc5fa84dff4d1100a482265f206ce6...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,8,48514,True,data/processed/images_silver/v1/pathology=Meni...,21021,630,630,224,224,1,v1,Meningioma,T1C+,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3952,849a7048a7f1145c95ae9b240001afdf9cb1ffbf9850aa...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,3,29189,True,data/processed/images_silver/v1/pathology=Gang...,20235,630,630,224,224,1,v1,Ganglioglioma,T1,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,val
3953,919564f89bfa4e9e5707d479c383464b84685f86f69d23...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,3,29654,True,data/processed/images_silver/v1/pathology=Gang...,20406,630,630,224,224,1,v1,Ganglioglioma,T1,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,val
3954,3791f57329d4357233b2e1170b956ddf0834e22af8a7d6...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,6,58981,True,data/processed/images_silver/v1/pathology=Gran...,28303,499,617,224,224,1,v1,Granuloma,T2,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,val
3955,910d5e7b6ee08db283684c6cac6c8bc3fbfb6fc5a1922b...,file:/home/dan/projects/ESGI/4A/T2/spark-core/...,6,50750,True,data/processed/images_silver/v1/pathology=Gran...,25689,502,630,224,224,1,v1,Granuloma,T2,split_seed42_r70_15_15_20260223T100118Z,42,0.7000/0.1500/0.1500,val


In [9]:
training_manifest = spark.read.parquet("../data/processed/splits/current/training_manifest.parquet")
display(training_manifest.select("label_idx").groupBy("label_idx").count().toPandas())


,label_idx,count
0,12,237
1,1,251
2,13,465
3,6,78
4,3,61
5,5,204
6,9,522
7,4,100
8,8,874
9,7,131


In [9]:
training_shards_versions = spark.read.parquet("../data/processed/training_shards_versions.parquet")
display(training_shards_versions.orderBy("generated_at_utc", ascending=False).toPandas().T)

,0
generated_at_utc,2026-02-23T08:11:56.039507+00:00
seed,42
n_shards,16
rows_total,3957
rows_train,2752
rows_val,591
rows_test,614
distinct_labels,14
input_manifest_path,data/processed/splits/current/training_manifes...
output_path,data/processed/training_shards/training_shards...


In [13]:
path = "../data/processed/training_shards/training_shards_seed42_n16_20260223T081146Z"

training_shards = spark.read.parquet(path)
display(training_shards.columns)
display(training_shards.toPandas())

['image_id',
 'processed_path',
 'label_idx',
 'split_id',
 'export_id',
 'shard_seed',
 'n_shards',
 'split',
 'shard_id']

,image_id,processed_path,label_idx,split_id,export_id,shard_seed,n_shards,split,shard_id
0,c890724a5def652cbe280d2befe567b20813fea2e867f0...,data/processed/images_silver/v1/pathology=Meni...,8,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,train,15
1,39de99ff0d7d7330835385283ea35ce91855d0df5623e8...,data/processed/images_silver/v1/pathology=Meni...,8,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,train,15
2,1c19da5e6327690fb15c2e4e68c39d3d91331537f272c8...,data/processed/images_silver/v1/pathology=Meni...,8,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,train,15
3,7e00936cf983e056ccad96e12dee5088a039beaba5c3b8...,data/processed/images_silver/v1/pathology=Meni...,8,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,train,15
4,dffeb95e9a82905be43bef669ab48d827bc702695a13fd...,data/processed/images_silver/v1/pathology=Meni...,8,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,train,15
...,...,...,...,...,...,...,...,...,...
3952,8922612cfcbba77612446f25373321ddcad49b1a947bb8...,data/processed/images_silver/v1/pathology=Olig...,10,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,val,7
3953,f2d4fc68d624145be9f344719b779549ccc01a161772f7...,data/processed/images_silver/v1/pathology=Papi...,11,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,val,7
3954,d3bf62c982390004e2168240d36240bb1c8c9214c4ea66...,data/processed/images_silver/v1/pathology=Glio...,5,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,val,7
3955,af719960832b333c02904b768f53dafabf93676668b837...,data/processed/images_silver/v1/pathology=Medu...,7,split_seed42_r70_15_15_20260223T080628Z,training_shards_seed42_n16_20260223T081146Z,42,16,val,7


In [4]:
from pathlib import Path
import tensorflow as tf
import pandas as pd

root = Path("../data/processed/training_tfrecord/current")
files = sorted(str(p) for p in root.glob("split=*/shard_id=*/*.tfrecord"))
print(f"{len(files)} fichiers TFRecord trouvés")

feature_spec = {
    "image_bytes": tf.io.FixedLenFeature([], tf.string),
    "label_idx": tf.io.FixedLenFeature([], tf.int64),
    "image_id": tf.io.FixedLenFeature([], tf.string),
    "split": tf.io.FixedLenFeature([], tf.string),
    "split_id": tf.io.FixedLenFeature([], tf.string),
    "shard_id": tf.io.FixedLenFeature([], tf.int64),
}

rows = []
max_rows = 200 
for raw in tf.data.TFRecordDataset(files):
    ex = tf.io.parse_single_example(raw, feature_spec)
    rows.append({
        "image_id": ex["image_id"].numpy().decode("utf-8"),
        "split": ex["split"].numpy().decode("utf-8"),
        "split_id": ex["split_id"].numpy().decode("utf-8"),
        "label_idx": int(ex["label_idx"].numpy()),
        "shard_id": int(ex["shard_id"].numpy()),
        "image_bytes_len": len(ex["image_bytes"].numpy()),
    })
    if len(rows) >= max_rows:
        break

df = pd.DataFrame(rows)
display(df)


2026-02-26 19:36:47.424693: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


48 fichiers TFRecord trouvés


I0000 00:00:1772131015.247608   19921 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1768 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
2026-02-26 19:36:55.512237: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144


,image_id,split,split_id,label_idx,shard_id,image_bytes_len
0,8d957d30a19fbd2955e739b040acfd63b12639ea3ca815...,test,split_seed42_r70_15_15_20260223T080628Z,8,0,19550
1,0704561b7f7533c6faf14aa85a0ddbc3403dd35335ca75...,test,split_seed42_r70_15_15_20260223T080628Z,8,0,21548
2,c4fd9b9cd44675ec70a5395f3280c7bb2da36ce0f85a76...,test,split_seed42_r70_15_15_20260223T080628Z,0,0,14617
3,6248b01b6daf2625ba871f1a8abaed339ffe2a234eb9ec...,test,split_seed42_r70_15_15_20260223T080628Z,0,0,20296
4,03689b4c01092df068a3e46cab5267b37270224f4930f7...,test,split_seed42_r70_15_15_20260223T080628Z,8,0,25919
...,...,...,...,...,...,...
195,5b0b4e3a98f86c980e1e15b7a0e924031f8eb3d538a54f...,test,split_seed42_r70_15_15_20260223T080628Z,12,13,21058
196,983c672f5bd172832ce662b403d693d890694f00686d9c...,test,split_seed42_r70_15_15_20260223T080628Z,12,13,21549
197,dec788e48e846304e2295c920e7a5e1f264befcfb4f124...,test,split_seed42_r70_15_15_20260223T080628Z,12,13,31311
198,1be0b8b3c40e19fb3f9a07ed1e903661c2da89cd83ee9a...,test,split_seed42_r70_15_15_20260223T080628Z,0,13,19423
